# Global Air Quality Index — Data Analysis
**Dataset:** Global Air Quality Dataset (10 cities, 2024)  
**Module:** DATA 2005 - Data-Centric Programming  
Air Quality (Domain 1)

---
## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')
print('Libraries loaded successfully')

---
## 2. Load Data

In [ ]:
df_raw = pd.read_csv('data/raw/global_air_quality_dataset.csv')

print('Shape:', df_raw.shape)
print('Columns:', df_raw.columns.tolist())
df_raw.head()

---
## 3. Preprocessing
Steps:
- Rename columns to simpler names
- Parse dates and extract month/year
- Convert measurements to numbers
- Handle missing values
- Cap outliers using IQR method

In [ ]:

renamed_columns = {
    'Date': 'date', 'City': 'city', 'Country': 'country',
    'AQI': 'aqi', 'PM2.5 (µg/m³)': 'pm25', 'PM10 (µg/m³)': 'pm10',
    'NO2 (ppb)': 'no2', 'SO2 (ppb)': 'so2', 'CO (ppm)': 'co',
    'O3 (ppb)': 'o3', 'Temperature (°C)': 'temperature',
    'Humidity (%)': 'humidity', 'Wind Speed (m/s)': 'wind_speed'
}
df = df_raw.rename(columns=renamed_columns)

# Parse dates
df['date']  = pd.to_datetime(df['date'], format='%Y-%m-%d')
df['month'] = df['date'].dt.month
df['year']  = df['date'].dt.year

# Convert to numeric
measurement_cols = ['aqi','pm25','pm10','no2','so2','co','o3','temperature','humidity','wind_speed']
for col in measurement_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Missing values
print('Missing values before:', df.isnull().sum().sum())
for col in measurement_cols:
    df[col] = df[col].fillna(df[col].median())
print('Missing values after: ', df.isnull().sum().sum())

In [ ]:
# Outlier handling (IQR method)
outlier_cols = ['aqi','pm25','pm10','no2','so2','co','o3']

for col in outlier_cols:
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers_found = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    print(f'{col}: {outliers_found} outliers capped')

print(f'\nDate range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Total rows: {len(df)}')
df.head()

---
## 4. Export Cleaned Data

In [ ]:
os.makedirs('data/processed', exist_ok=True)

df.to_csv('data/processed/Air_Quality_clean.csv', index=False)
print('Exported: Air_Quality_clean.csv')

df.to_json('data/processed/Air_Quality_clean.json', orient='records', date_format='iso', indent=2)
print('Exported: Air_Quality_clean.json')

---
## 5. Descriptive Statistics

In [ ]:
stats_cols = ['aqi','pm25','pm10','no2','so2','co','o3']

stats = {}
for col in stats_cols:
    values = df[col].dropna()
    stats[col] = {
        'mean':   round(np.mean(values), 2),
        'median': round(np.median(values), 2),
        'std':    round(np.std(values, ddof=1), 2),
        'var':    round(np.var(values, ddof=1), 2),
        'p25':    round(np.percentile(values, 25), 2),
        'p75':    round(np.percentile(values, 75), 2),
        'min':    round(np.min(values), 2),
        'max':    round(np.max(values), 2)
    }

stats_df = pd.DataFrame(stats).T
print('Descriptive Statistics:')
stats_df

In [ ]:
# Average AQI by City
print('Average AQI by City:')
df.groupby('city')['aqi'].mean().round(2).sort_values(ascending=False)

In [ ]:
# Monthly averages
print('Monthly Averages:')
df.groupby('month')[['aqi','pm25','pm10']].mean().round(2)

---
## 6. Visualisations

In [ ]:
# Plot 1 — Average AQI by City
city_aqi = df.groupby('city')['aqi'].mean().sort_values(ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=city_aqi, x='city', y='aqi', palette='Reds_d', ax=ax)
ax.set_title('Average AQI by City', fontsize=14)
ax.set_xlabel('City')
ax.set_ylabel('Mean AQI')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2 — Monthly AQI Trend
MONTH_ORDER = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df['month_name'] = pd.Categorical(df['date'].dt.strftime('%b'), categories=MONTH_ORDER, ordered=True)
monthly = df.groupby('month_name', observed=True)['aqi'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly['month_name'], monthly['aqi'], color='tomato', marker='o', linewidth=2)
ax.set_title('Average AQI by Month', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Mean AQI')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3 — PM2.5 Distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['pm25'].dropna(), bins=30, color='steelblue', edgecolor='white')
ax.axvline(15, color='red', linestyle='--', label='WHO guideline (15 µg/m³)')
ax.set_title('PM2.5 Distribution', fontsize=14)
ax.set_xlabel('PM2.5 (µg/m³)')
ax.set_ylabel('Number of Readings')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot 4 — 30-Day Rolling Average AQI (Top 3 Cities)
top3 = df.groupby('city')['aqi'].mean().nlargest(3).index.tolist()
df_top3 = df[df['city'].isin(top3)].sort_values('date')

fig, ax = plt.subplots(figsize=(12, 5))
for city in top3:
    city_df = df_top3[df_top3['city'] == city].set_index('date')
    rolling = city_df['aqi'].rolling(window=30, min_periods=1).mean()
    ax.plot(rolling.index, rolling.values, linewidth=2, label=city)

ax.set_title('30-Day Rolling Average AQI — Top 3 Most Polluted Cities', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('AQI (30-day rolling mean)')
ax.legend()
plt.tight_layout()
plt.show()